In [1]:
import pandas as pd
from huggingface_hub import hf_hub_download
import os
pd.set_option('display.max_columns', None)

In [2]:
caminho_diesel = hf_hub_download(
    repo_id="pbf-feijao-mecai-usp/bf-feijao-dados",
    filename="interim/diesel/diesel_deflacionado.parquet",
    repo_type="dataset"
)


caminho_safra = hf_hub_download(
repo_id="pbf-feijao-mecai-usp/bf-feijao-dados",
filename="interim/safra/producao_feijao_tons.parquet",
repo_type="dataset"
)

caminho_precos = hf_hub_download(
    repo_id="pbf-feijao-mecai-usp/bf-feijao-dados",
    filename="raw/painel_feijao_deflacionado.csv",
    repo_type="dataset"
)



caminho_repasse = hf_hub_download(
    repo_id="pbf-feijao-mecai-usp/bf-feijao-dados",
    filename="raw/Painel_Unificado_2004_2026_Deflacionado.xlsx",
    repo_type="dataset"
)

datasets = {
    "diesel": caminho_diesel,
    "safra": caminho_safra,
    "precos": caminho_precos,
    "bolsa_familia": caminho_repasse
}

In [3]:
datasets_carregados = {}

for nome, caminho in datasets.items():
    if caminho.endswith(".parquet"):
        df_temp = pd.read_parquet(caminho)
    elif  caminho.endswith(".csv"):
        df_temp = pd.read_csv(caminho)
    elif caminho.endswith(".xlsx"):
        df_temp = pd.read_excel(caminho)
    else:
        continue
    datasets_carregados[nome] = df_temp

In [4]:
df_precos = datasets_carregados["precos"]
df_safra = datasets_carregados["safra"]
df_diesel = datasets_carregados["diesel"]
df_repasse = datasets_carregados["bolsa_familia"]

### Tratamentos indiduais das tabelas para construir o dataset unificado

In [5]:
dic_estado_sigla = {
    # Norte
    "acre": "AC",
    "amapa": "AP",
    "amazonas": "AM",
    "para": "PA",
    "rondonia": "RO",
    "roraima": "RR",
    "tocantins": "TO",
    
    # Nordeste
    "alagoas": "AL",
    "bahia": "BA",
    "ceara": "CE",
    "maranhao": "MA",
    "paraiba": "PB",
    "pernambuco": "PE",
    "piaui": "PI",
    "rio_grande_do_norte": "RN",
    "sergipe": "SE",
    
    # Centro-Oeste
    "distrito_federal": "DF",
    "goias": "GO",
    "mato_grosso": "MT",
    "mato_grosso_do_sul": "MS",
    
    # Sudeste
    "espirito_santo": "ES",
    "minas_gerais": "MG",
    "rio_de_janeiro": "RJ",
    "sao_paulo": "SP",
    
    # Sul
    "parana": "PR",
    "rio_grande_do_sul": "RS",
    "santa_catarina": "SC"
}

In [6]:
# Tratamentos de SAFRA

data_safra = df_safra["ano"].astype(str) + "-" + df_safra["mes"].astype(str) + "-01"
df_safra["data_referencia"] = pd.to_datetime(data_safra)

df_safra_agrupado = df_safra.groupby(["data_referencia","uf"])["quantidade_tons"].sum().reset_index()
df_safra_agrupado["uf"] = df_safra_agrupado["uf"].map(dic_estado_sigla)

df_safra_agrupado.rename(columns={"quantidade_tons": "produção_feijao_tons"}, inplace=True )

display(df_safra_agrupado.head())
len(df_safra_agrupado)

,data_referencia,uf,produção_feijao_tons
0,2006-09-01,AC,8267
1,2006-09-01,AL,49735
2,2006-09-01,AP,635
3,2006-09-01,AM,3719
4,2006-09-01,BA,352142


6453

In [7]:
dic_capital_formatada = {
    # Norte
    "RIO BRANCO": "Rio Branco",
    "MACAPA": "Macapá",
    "MANAUS": "Manaus",
    "BELEM": "Belém",
    "PORTO VELHO": "Porto Velho",
    "BOA VISTA": "Boa Vista",
    "PALMAS": "Palmas",
    
    # Nordeste
    "MACEIO": "Maceió",
    "SALVADOR": "Salvador",
    "FORTALEZA": "Fortaleza",
    "SAO LUIS": "São Luís",
    "JOAO PESSOA": "João Pessoa",
    "RECIFE": "Recife",
    "TERESINA": "Teresina",
    "NATAL": "Natal",
    "ARACAJU": "Aracaju",
    
    # Centro-Oeste
    "BRASILIA": "Brasília",
    "GOIANIA": "Goiânia",
    "CUIABA": "Cuiabá",
    "CAMPO GRANDE": "Campo Grande",
    
    # Sudeste
    "VITORIA": "Vitória",
    "BELO HORIZONTE": "Belo Horizonte",
    "RIO DE JANEIRO": "Rio de Janeiro",
    "SAO PAULO": "São Paulo",
    
    # Sul
    "CURITIBA": "Curitiba",
    "PORTO ALEGRE": "Porto Alegre",
    "FLORIANOPOLIS": "Florianópolis"
}

In [8]:
# Tratamentos do DIESEL
df_diesel["data_referencia"] = df_diesel["data_da_coleta"].dt.to_period("M").dt.to_timestamp()
df_diesel["year"] = df_diesel["data_da_coleta"].dt.year
df_diesel["month"] = df_diesel["data_da_coleta"].dt.month
capitais = ["ARACAJU", "BELEM", "BELO HORIZONTE", "BRASILIA", "CURITIBA", "FORTALEZA", "GOIANIA", "JOAO PESSOA", "NATAL", "PORTO ALEGRE", "RECIFE", "RIO DE JANEIRO", "SALVADOR", "SAO PAULO", "VITORIA", "FLORIANOPOLIS"]

df_diesel = df_diesel[df_diesel['municipio'].isin(capitais)]

df_diesel_agrupado = df_diesel.pivot_table(
    index=["data_referencia", "municipio"],
    columns="produto",
    values=["valor_de_venda", "valor_de_venda_real"],
    aggfunc="mean"
)

df_diesel_agrupado.columns = [f"{andar0}_{andar1}" for andar0, andar1 in df_diesel_agrupado.columns]
df_diesel_agrupado = df_diesel_agrupado.reset_index()

cols_venda = ['valor_de_venda_real_DIESEL', 'valor_de_venda_real_diesel_s50', 'valor_de_venda_real_diesel_s10']
cols_venda_existentes = [col for col in cols_venda if col in df_diesel_agrupado.columns]
df_diesel_agrupado['venda_real_diesel_media_geral'] = df_diesel_agrupado[cols_venda_existentes].astype(float).mean(axis=1, skipna=True)

cols_nom = ['valor_de_venda_DIESEL', 'valor_de_venda_diesel_s50', 'valor_de_venda_diesel_s10']
cols_nom_existentes = [col for col in cols_nom if col in df_diesel_agrupado.columns]
df_diesel_agrupado['valor_venda_diesel_media_geral'] = df_diesel_agrupado[cols_nom_existentes].astype(float).mean(axis=1, skipna=True)

df_diesel_agrupado.rename(
    columns={
        'municipio': 'capital',
        'valor_de_venda_real_DIESEL': 'venda_real_diesel_comum',
        'valor_de_venda_DIESEL': 'valor_venda_diesel_comum'
    }, 
    inplace=True
)

df_diesel_agrupado["capital"] = df_diesel_agrupado["capital"].map(dic_capital_formatada)


display(df_diesel_agrupado.head())
len(df_diesel_agrupado)

,data_referencia,capital,valor_venda_diesel_comum,valor_de_venda_DIESEL S10,valor_de_venda_DIESEL S50,venda_real_diesel_comum,valor_de_venda_real_DIESEL S10,valor_de_venda_real_DIESEL S50,venda_real_diesel_media_geral,valor_venda_diesel_media_geral
0,2004-05-01,Aracaju,1.335657,NaN,NaN,4.450634,NaN,NaN,4.450634,1.335657
1,2004-05-01,Belém,1.427871,NaN,NaN,4.757907,NaN,NaN,4.757907,1.427871
2,2004-05-01,Belo Horizonte,1.369720,NaN,NaN,4.564138,NaN,NaN,4.564138,1.369720
3,2004-05-01,Brasília,1.452098,NaN,NaN,4.838634,NaN,NaN,4.838634,1.452098
4,2004-05-01,Curitiba,1.380254,NaN,NaN,4.599237,NaN,NaN,4.599237,1.380254


4216

In [9]:
# Tratamentos de PREÇO DO FEIJÂO

df_precos["data_referencia"] = pd.to_datetime(df_precos["periodo"], format="%m/%Y")
df_precos.drop(columns=["periodo","ano","mes", "t"], inplace=True)
df_precos.rename(columns={"preco_nominal": "preco_nominal_feijao",
                          "preco_real":"preco_real_feijao",
                          "log_preco_real":"log_preco_feijao_real",
                          "var_pct_real":"var_pct_ipca_real"
                          "log_preco_nominal":"log_preco_feijao_nominal"
                          }, inplace=True)
display(df_precos.head())
len(df_precos)

SyntaxError: invalid syntax (1710215974.py, line 9)

In [ ]:
# Tratamentos de REPASSE DO BOLSA FAMÍLIA

df_repasse["data_referencia"] = pd.to_datetime(df_repasse["periodo"], format="%Y%m")
df_repasse.rename(columns={
                           "codigo":"codigo_cidade",
                           "cidade":"capital",
                           "valor_repassado":"valor_repassado_pbf",
                           "valor_repassado_real":"valor_repassado_pbf_real",
                           "ln_valor_repassado_real":"ln_valor_repassado_pbf_real"
                           }, inplace=True)
df_repasse.drop(columns="periodo", inplace=True)
display(df_repasse.head())
len(df_repasse)

,codigo_cidade,capital,programa,qtd_familias,valor_repassado_pbf,ipca_indice,valor_repassado_pbf_real,ln_valor_repassado_pbf_real,data_referencia
0,150140,Belém,Bolsa Família,298,20865,100.000000,20865.00,9.945828,2004-01-01
1,150140,Belém,Bolsa Família,298,20865,100.690000,20722.02,9.938952,2004-02-01
2,150140,Belém,Bolsa Família,298,20865,101.163243,20625.08,9.934263,2004-03-01
3,150140,Belém,Bolsa Família,369,24380,101.537547,24010.82,10.086260,2004-04-01
4,150140,Belém,Bolsa Família,11026,686140,102.055388,672321.19,13.418491,2004-05-01


4352

In [ ]:
# Alinhando todos os textos que serão indices do merge
def padronizar_texto(coluna_pandas):
    return (
        coluna_pandas
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
        .str.normalize("NFKD")
        .str.encode("ascii", errors="ignore")
        .str.decode("utf-8")
    )

df_precos['capital_join'] = padronizar_texto(df_precos['capital'])
df_repasse['capital_join'] = padronizar_texto(df_repasse['capital'])
df_diesel_agrupado['capital_join'] = padronizar_texto(df_diesel_agrupado['capital'])

df_precos['uf_join'] = df_precos['uf'].str.upper().str.strip()
df_safra_agrupado['uf_join'] = df_safra_agrupado['uf'].str.upper().str.strip()

In [ ]:
# unindo as tabelas
df_unificado = df_precos.merge(df_safra_agrupado, on= ["data_referencia","uf"], how="left") \
    .merge(df_diesel_agrupado, on= ["data_referencia","capital"], how="left") \
    .merge(df_repasse, on= ["data_referencia","capital"], how="left")

In [ ]:
# descartando colunas auxiliares
colunas_join_descartar = [col for col in df_unificado.columns if "_join" in col]
colunas_y_descartar = [col for col in df_unificado.columns if col.endswith("_y")]
df_unificado.drop(columns=colunas_join_descartar + colunas_y_descartar, inplace= True)
df_unificado.columns = df_unificado.columns.str.replace("_x","")

In [ ]:
display(df_unificado.sample(5))
len(df_unificado)

,capital_id,capital,uf,regiao,preco_nominal_feijao,preco_real_feijao,log_preco_feijao_real,ipca_var_pct,ipca_indice,ln_ipca_indice,var_pct_real,log_preco_feijao_nominal,data_referencia,produção_feijao_tons,valor_venda_diesel_comum,valor_de_venda_DIESEL S10,valor_de_venda_DIESEL S50,venda_real_diesel_comum,valor_de_venda_real_DIESEL S10,valor_de_venda_real_DIESEL S50,venda_real_diesel_media_geral,valor_venda_diesel_media_geral,codigo_cidade,programa,qtd_familias,valor_repassado_pbf,valor_repassado_pbf_real,ln_valor_repassado_pbf_real
2566,10,Belém,PA,Norte,4.11,2.3227,0.842730,0.25,176.9475,1.018658,-13.5096,1.413423,2014-08-01,29089.0,NaN,2.631442,NaN,NaN,5.061307,NaN,5.061307,2.631442,150140,Bolsa Família,101459,15271359,8630444.41,15.970807
4099,16,Salvador,BA,Nordeste,1.85,1.5938,0.466121,0.31,116.0742,0.770452,5.9919,0.615186,2006-11-01,332178.0,1.814804,NaN,NaN,5.322764,NaN,NaN,5.322764,1.814804,292740,Bolsa Família,125386,7287753,6278531.42,15.652647
2517,10,Belém,PA,Norte,3.40,2.4504,0.896251,0.01,138.7526,0.870258,-3.4173,1.223775,2010-07-01,28000.0,2.015689,NaN,NaN,4.945668,NaN,NaN,4.945668,2.015689,150140,Bolsa Família,75422,7416297,5344978.04,15.491668
1265,5,São Paulo,SP,Sudeste,7.56,3.3136,1.198035,0.43,228.1475,1.188293,48.4721,2.022871,2019-02-01,255656.0,3.236859,3.472206,NaN,4.807944,5.157522,NaN,4.982733,3.354532,355030,Bolsa Família,472099,73800539,32347736.41,17.292055
3092,12,Fortaleza,CE,Nordeste,5.87,3.5643,1.270968,0.55,164.6906,0.973391,4.9991,1.769855,2013-04-01,158576.0,NaN,2.340587,NaN,NaN,4.838363,NaN,4.838363,2.340587,230440,Bolsa Família,190413,26005901,15790763.57,16.574936


4336

In [ ]:
def polimento(self):
    
    colunas_duplicadas = [col for col in self.columns if col.endswith('_y')]
    self.drop(columns=colunas_duplicadas, inplace=True)
    self.columns = self.columns.str.replace('_x', '')

    return(self)


In [ ]:
dataset_final = polimento(df_unificado)

In [ ]:
# dataset final
dataset_final.sample(3)

,capital_id,capital,uf,regiao,preco_nominal_feijao,preco_real_feijao,log_preco_feijao_real,ipca_var_pct,ipca_indice,ln_ipca_indice,var_pct_real,log_preco_feijao_nominal,data_referencia,produção_feijao_tons,valor_venda_diesel_comum,valor_de_venda_DIESEL S10,valor_de_venda_DIESEL S50,venda_real_diesel_comum,valor_de_venda_real_DIESEL S10,valor_de_venda_real_DIESEL S50,venda_real_diesel_media_geral,valor_venda_diesel_media_geral,codigo_cidade,programa,qtd_familias,valor_repassado_pbf,valor_repassado_pbf_real,ln_valor_repassado_pbf_real
3451,13,João Pessoa,PB,Nordeste,7.77,3.2668,1.183811,0.24,237.8488,1.217428,-8.5903,2.050270,2020-08-01,36046.0,3.189143,3.321776,NaN,4.543844,4.732817,NaN,4.638330,3.255459,250750,Bolsa Família,61127,579219,243524.02,12.402971
1091,5,São Paulo,SP,Sudeste,2.31,2.2120,0.793897,0.69,104.4309,0.715060,-7.1174,0.837248,2004-08-01,NaN,1.495247,NaN,NaN,4.869101,NaN,NaN,4.869101,1.495247,355030,Bolsa Família,147371,8828580,8453990.85,15.950149
667,3,Belo Horizonte,MG,Sudeste,3.99,2.2608,0.815719,0.40,176.4886,1.016999,-12.0756,1.383791,2014-06-01,597672.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,310620,Bolsa Família,67585,10282777,5826311.48,15.577895
